# Code for `speeches.csv` data
This document provides the code used to create a dataframe of speeches, with columns including the `year` of the speech, the `speech_text`, and the `link` for the speech page.

Please see the main project ipynb file for further transformation and analysis of the `speeches.csv` data.

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

## Step 1: Get all the urls for each State of the Union Speech.

The American Presidency Project provides transcripts of all State of the Union Addresses here: https://www.presidency.ucsb.edu/documents/presidential-documents-archive-guidebook/annual-messages-congress-the-state-the-union#Table%20of%20SOTU.

The code in the section below collects all urls to the transcript pages, which will later be fed into a loop to create a dataframe of speeches.

In [2]:
# url for table of all speeches
r = requests.get("https://www.presidency.ucsb.edu/documents/presidential-documents-archive-guidebook/annual-messages-congress-the-state-the-union")
r.status_code

200

In [3]:
# getting table information from html text
soup = BeautifulSoup(r.text)
sotu_table = soup.find_all('table')[0]
sotu_rows = sotu_table.find_all('tr')

In [4]:
# creating a list of all speech urls

sotu_links = []

for row in sotu_rows[1:]:
    link_cols = row.find_all('td')

    for col in link_cols[2:6]:
        link = col.find('a')
        if link and link.get("href"):
          sotu_links.append(link.get("href"))

In [5]:
# getting the 45 most recent speeches

last_45_speeches = sotu_links[:45]

- Retrieved the most recent 45 State of the Union Addresses.

## Step 2: Create a table with `year` `speech_text`, `total_words` and `link` for each speech.

The code in the section below scrapes the html text from each page to retrieve the year the speech was made, the speech text, the total word count for each speech, and the link for the speech page.

First, a dataframe with year, speech text, and link:


In [ ]:
speeches_data = []

for speech_link in last_45_speeches:
    html_text = requests.get(speech_link)
    assert html_text.status_code == 200

    page_soup = BeautifulSoup(html_text.text)

    # speech_text
    html_speech_text = page_soup.find('div', class_ = "field-docs-content")
    clean_speech = html_speech_text.get_text(strip=True).lower() # this is the cleaned text of the speech

    # president
    year = (page_soup.find('div', class_ = "field-docs-start-date-time").find('span').get('content'))[:4]

    if clean_speech:
        speeches_data.append({'year': year, 'speech_text': clean_speech, 'link': speech_link})

In [ ]:
# create the df from the dictionary
speech_df = pd.DataFrame(speeches_data)

Next, a dataframe of year and word count:

(The American Presidency Project provides an untidy table of word counts for each speech. Located here: https://www.presidency.ucsb.edu/documents/presidential-documents-archive-guidebook/annual-messages-congress-the-state-the-union-3).

In [ ]:
# scraping the page for the speech lengths table
length_html = requests.get("https://www.presidency.ucsb.edu/documents/presidential-documents-archive-guidebook/annual-messages-congress-the-state-the-union-3")
length_html.status_code

In [ ]:
# getting the last 53 rows which contain data for the most recent 45 speeches
counts_soup = BeautifulSoup(length_html.text)
counts_table = counts_soup.find('table')
counts_rows = counts_table.find_all('tr')
last_53 = counts_rows[-53:]

In [ ]:
# a loop to create a df of speech length data

speech_lengths = []
for row in last_53:
    row_td_tags = row.find_all('td')

    if len(row_td_tags) >= 4:
        year = row_td_tags[1].get_text(strip=True)[-4:]
        word_count = row_td_tags[3].get_text(strip=True)
        speech_lengths.append({'year': year, 'word_count': word_count})
    else:
        pass

lengths_df = pd.DataFrame(speech_lengths)

Now, we join the `speech_df` and `lengths_df`

In [ ]:
joined_df = pd.merge(speech_df, lengths_df, on='year', how='inner')

## Step 3: Downloading the dataframe as a file:

In [ ]:
# for downloading the csv file
joined_df.to_csv('speeches45.csv', index=False)

***Next***: Please see the main project ipynb file for further transformation and analysis of the data.